# PathRAG Full-System Colab Run

Run this notebook top-to-bottom on a **GPU runtime**. Use an A100 or similar if you want local `llava-med` to be stable.

This notebook does four things:
1. clones `pathrag-agentic-starter`
2. creates dedicated tool environments for histocartography and CHIEF with **Python 3.9**
3. installs a local `llava-med` runtime in Colab
4. runs smoke checks and then a **full graph run**

What you still need to provide:
- a slide or flat pathology image path
- CHIEF source tree
- CHIEF model weights
- Hugging Face access to `microsoft/llava-med-v1.5-mistral-7b`
- optional `OPENAI_API_KEY` if you want real Stage 5 / Stage 7

## 0. Runtime choice

Before running the notebook:
- `Runtime` -> `Change runtime type`
- Hardware accelerator: `GPU`
- Preferred: `A100`

The notebook keeps the **main orchestration** in the Colab kernel and isolates these heavy tools in separate environments:
- histocartography: Python 3.9
- CHIEF: Python 3.9
- LLaVA-Med: Python 3.10

In [ ]:
from pathlib import Path
import os

# Edit these before running the rest of the notebook.
BRANCH = "local-chief"
REPO_URL = "https://github.com/embedded-robotics/path-agent.git"
WORKDIR = Path("/content/path-agent")
STARTER_DIR = WORKDIR / "pathrag-agentic-starter"

# Input image + question for the full graph run.
# Use a flat image (.jpg/.png) or a slide path (.svs).
IMAGE_PATH = "/content/drive/MyDrive/pathrag_inputs/07_level2.jpg"
QUESTION = "What are the prominent, discrete, thick-walled circular structures with clear or collapsed lumens embedded within the looser pale pink stroma?"
TOP_K = 3
MAX_ROUNDS = 1

# Where CHIEF source + weights live in Drive.
DRIVE_CHIEF_REPO_DIR = "/content/drive/MyDrive/pathrag_assets/CHIEF/repo"
DRIVE_CHIEF_MODEL_DIR = "/content/drive/MyDrive/pathrag_assets/CHIEF/model_weight"

# LLaVA-Med setup.
LLAVA_REPO_URL = "https://github.com/microsoft/LLaVA-Med.git"
LLAVA_REPO_DIR = Path("/content/third_party/LLaVA-Med")
LLAVA_MODEL = "microsoft/llava-med-v1.5-mistral-7b"
LLAVA_CONV_MODE = "mistral_instruct"

# Optional: Stage 5 / Stage 7 real OpenAI path.
OPENAI_API_KEY = ""

for key, value in {
    "BRANCH": BRANCH,
    "REPO_URL": REPO_URL,
    "WORKDIR": str(WORKDIR),
    "STARTER_DIR": str(STARTER_DIR),
    "IMAGE_PATH": IMAGE_PATH,
    "QUESTION": QUESTION,
    "TOP_K": str(TOP_K),
    "MAX_ROUNDS": str(MAX_ROUNDS),
    "DRIVE_CHIEF_REPO_DIR": DRIVE_CHIEF_REPO_DIR,
    "DRIVE_CHIEF_MODEL_DIR": DRIVE_CHIEF_MODEL_DIR,
    "LLAVA_REPO_URL": LLAVA_REPO_URL,
    "LLAVA_REPO_DIR": str(LLAVA_REPO_DIR),
    "LLAVA_MODEL": LLAVA_MODEL,
    "LLAVA_CONV_MODE": LLAVA_CONV_MODE,
}.items():
    os.environ[key] = value

print({
    "branch": BRANCH,
    "repo_url": REPO_URL,
    "image_path": IMAGE_PATH,
    "llava_model": LLAVA_MODEL,
})


{'branch': 'local-chief', 'repo_url': 'https://github.com/embedded-robotics/path-agent.git', 'image_path': '/content/drive/MyDrive/pathrag_inputs/07_level2.jpg', 'llava_model': 'microsoft/llava-med-v1.5-mistral-7b'}


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%%bash
set -euo pipefail

apt-get update -y
apt-get install -y git git-lfs rsync curl wget build-essential libgl1 libglib2.0-0 libopenslide0 openslide-tools

if [ ! -x /content/bin/micromamba ]; then
  mkdir -p /content/bin
  cd /content
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

/content/bin/micromamba --help >/dev/null

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
libgl1 is already the newest version (1.4.0-1).
libopenslide0 is 

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
set -euo pipefail

rm -rf /content/path-agent
mkdir -p /content

git clone "$REPO_URL" /content/path-agent
cd /content/path-agent
git checkout "$BRANCH"

Branch 'local-chief' set up to track remote branch 'local-chief' from 'origin'.


Cloning into '/content/path-agent'...
Switched to a new branch 'local-chief'


In [ ]:
import pathlib
import subprocess
import sys

starter_reqs = STARTER_DIR / "requirements.txt"
filtered = []
for line in starter_reqs.read_text(encoding="utf-8").splitlines():
    raw = line.strip()
    if not raw:
        filtered.append(line)
        continue
    if raw.startswith("--extra-index-url"):
        continue
    if raw.startswith("torch"):
        continue
    if raw.startswith("torchvision"):
        continue
    filtered.append(line)

main_req = pathlib.Path("/content/pathrag-main-requirements.txt")
main_req.write_text("\n".join(filtered) + "\n", encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(main_req)])
print("Main runtime ready")


Main runtime ready


In [ ]:
%%bash
set -euo pipefail

mkdir -p ~/.local/share/pathrag/chief
mkdir -p ~/.local/share/pathrag/histocartography/checkpoints
rm -rf ~/.local/share/pathrag/chief/repo ~/.local/share/pathrag/chief/model_weight
mkdir -p ~/.local/share/pathrag/chief/repo ~/.local/share/pathrag/chief/model_weight

if [ ! -d "$DRIVE_CHIEF_REPO_DIR" ]; then
  echo "Missing CHIEF repo dir: $DRIVE_CHIEF_REPO_DIR" >&2
  exit 1
fi
if [ ! -d "$DRIVE_CHIEF_MODEL_DIR" ]; then
  echo "Missing CHIEF model dir: $DRIVE_CHIEF_MODEL_DIR" >&2
  exit 1
fi

rsync -a "$DRIVE_CHIEF_REPO_DIR"/ ~/.local/share/pathrag/chief/repo/
rsync -a "$DRIVE_CHIEF_MODEL_DIR"/ ~/.local/share/pathrag/chief/model_weight/

ls -la ~/.local/share/pathrag/chief
ls -la ~/.local/share/pathrag/chief/model_weight

total 16
drwxr-xr-x 4 root root 4096 Apr 24 05:09 .
drwxr-xr-x 4 root root 4096 Apr 24 04:33 ..
drwx------ 2 root root 4096 Apr 20 02:05 model_weight
drwx------ 7 root root 4096 Apr 24 05:00 repo
total 116020
drwx------ 2 root root      4096 Apr 20 02:05 .
drwxr-xr-x 4 root root      4096 Apr 24 05:09 ..
-rw------- 1 root root 111292151 Aug  1  2025 CHIEF_CTransPath.pth
-rw------- 1 root root   2632007 Aug  1  2025 CHIEF_finetune.pth
-rw------- 1 root root   4808050 Aug  1  2025 CHIEF_pretraining.pth
-rw------- 1 root root     59115 Aug  1  2025 Text_emdding.pth


In [ ]:
%%bash
set -euo pipefail

MAMBA=/content/bin/micromamba
HC_ENV=/content/envs/histocartography39
CHIEF_ENV=/content/envs/chief39
LLAVA_ENV=/content/envs/llava310

$MAMBA create -y -p "$HC_ENV" python=3.9 pip
$MAMBA create -y -p "$CHIEF_ENV" python=3.9 pip
$MAMBA create -y -p "$LLAVA_ENV" python=3.10 pip

$MAMBA run -p "$HC_ENV" python --version
$MAMBA run -p "$CHIEF_ENV" python --version
$MAMBA run -p "$LLAVA_ENV" python --version



Transaction

  Prefix: /content/envs/histocartography39

  Updating specs:

   - python=3.9
   - pip


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + bzip2                 1.0.8  hda65f42_9            conda-forge     Cached
  + ca-certificates   2026.4.22  hbd8a1cb_0            conda-forge     Cached
  + icu                    78.3  h33c6efd_0            conda-forge     Cached
  + ld_impl_linux-64     2.45.1  default_hbd61a6d_102  conda-forge     Cached
  + libexpat              2.7.5  hecca717_0            conda-forge     Cached
  + libffi                3.5.2  h3435931_0            conda-forge     Cached
  + libgcc               15.2.0  he0feb66_18           conda-forge     Cached
  + libgcc-ng        

In [ ]:
%%bash
set -euo pipefail

MAMBA=/content/bin/micromamba
HC_ENV=/content/envs/histocartography39
CHIEF_ENV=/content/envs/chief39
LLAVA_ENV=/content/envs/llava310

# GPU-enabled torch stacks for the tool envs.
$MAMBA run -p "$HC_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
$MAMBA run -p "$CHIEF_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
$MAMBA run -p "$LLAVA_ENV" pip install -q --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio

# Tool-specific deps.
$MAMBA run -p "$HC_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/histocartography/requirements.txt
$MAMBA run -p "$CHIEF_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/chief/requirements.txt timm==0.5.4 rich typer httpx huggingface_hub
$MAMBA run -p "$LLAVA_ENV" pip install -q -r /content/path-agent/pathrag-agentic-starter/tools/llava_med/requirements.txt

In [ ]:
%%bash
set -euo pipefail

if [ ! -d "$LLAVA_REPO_DIR/.git" ]; then
  rm -rf "$LLAVA_REPO_DIR"
  mkdir -p "$(dirname "$LLAVA_REPO_DIR")"
  git clone "$LLAVA_REPO_URL" "$LLAVA_REPO_DIR"
fi

if [ -f "$LLAVA_REPO_DIR/requirements.txt" ]; then
  /content/bin/micromamba run -p /content/envs/llava310 pip install -q -r "$LLAVA_REPO_DIR/requirements.txt"
fi


In [ ]:
import getpass
import os
import subprocess

hf_token = getpass.getpass("Hugging Face token for LLaVA-Med (required if gated): ")
if hf_token:
    subprocess.check_call([
        "/content/bin/micromamba", "run", "-p", "/content/envs/llava310",
        "huggingface-cli", "login", "--token", hf_token,
    ])

if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("OpenAI API key (optional, press Enter to skip): ")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

Hugging Face token for LLaVA-Med (required if gated): ··········
OpenAI API key (optional, press Enter to skip): ··········


In [ ]:
import os

os.environ["PYTHONPATH"] = str(STARTER_DIR / "src")
os.environ["HISTOCARTOGRAPHY_PYTHON"] = "/content/envs/histocartography39/bin/python"
os.environ["CHIEF_PYTHON"] = "/content/envs/chief39/bin/python"
os.environ["PATHRAG_STAGE4_BACKEND"] = "llava-med"
os.environ.pop("PATHRAG_STAGE4_REMOTE_URL", None)
os.environ["USE_LLAVA"] = "1"
os.environ["LLMED_REPO"] = str(LLAVA_REPO_DIR)
os.environ["LLMED_PYTHON"] = "/content/envs/llava310/bin/python"
os.environ["LLMED_MODEL"] = LLAVA_MODEL
os.environ["LLMED_CONV_MODE"] = LLAVA_CONV_MODE
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")

for key in [
    "HISTOCARTOGRAPHY_PYTHON",
    "CHIEF_PYTHON",
    "PATHRAG_STAGE4_BACKEND",
    "USE_LLAVA",
    "LLMED_REPO",
    "LLMED_PYTHON",
    "LLMED_MODEL",
]:
    print(f"{key}={os.environ[key]}")

HISTOCARTOGRAPHY_PYTHON=/content/envs/histocartography39/bin/python
CHIEF_PYTHON=/content/envs/chief39/bin/python
PATHRAG_STAGE4_BACKEND=llava-med
USE_LLAVA=1
LLMED_REPO=/content/third_party/LLaVA-Med
LLMED_PYTHON=/content/envs/llava310/bin/python
LLMED_MODEL=microsoft/llava-med-v1.5-mistral-7b


In [ ]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/pathrag_assets/histocartography_checkpoints")
dst = Path("/root/.local/share/pathrag/histocartography/checkpoints")
dst.mkdir(parents=True, exist_ok=True)

for name in ["pannuke.pt", "monusac.pt"]:
    shutil.copy2(src / name, dst / name)

print(list(dst.glob("*")))



[PosixPath('/root/.local/share/pathrag/histocartography/checkpoints/pannuke.pt'), PosixPath('/root/.local/share/pathrag/histocartography/checkpoints/monusac.pt')]


In [ ]:
import json
import os
import subprocess
from pathlib import Path

hc_cmd = [
    os.environ["HISTOCARTOGRAPHY_PYTHON"],
    "-m", "src.graph.hc_eval",
    "--config", str(STARTER_DIR / "tools" / "histocartography" / "config" / "default.yaml"),
    "--image-path", IMAGE_PATH,
    "--top-n", str(TOP_K),
    "--out", "/content/hc_smoke.json",
]

try:
    result = subprocess.run(hc_cmd, cwd=str(STARTER_DIR / "tools" / "histocartography"), capture_output=True, text=True, check=True)
    print(Path("/content/hc_smoke.json").read_text())
except subprocess.CalledProcessError as e:
    print(f"Subprocess failed with exit code {e.returncode}")
    print("Stderr:")
    print(e.stderr)
    print("Stdout:")
    print(e.stdout)
    raise e

{
  "success": true,
  "image_path": "/content/drive/MyDrive/pathrag_inputs/07_level2.jpg",
  "slide_level": 0,
  "selected_patches": [
    {
      "patch_number": 1,
      "x1": 1228,
      "y1": 0,
      "x2": 2456,
      "y2": 853,
      "nuclei_count": 18
    },
    {
      "patch_number": 2,
      "x1": 0,
      "y1": 1706,
      "x2": 1228,
      "y2": 2560,
      "nuclei_count": 17
    },
    {
      "patch_number": 3,
      "x1": 0,
      "y1": 853,
      "x2": 1228,
      "y2": 1706,
      "nuclei_count": 16
    }
  ],
  "non_selected_patches": [
    {
      "x1": 2456,
      "y1": 853,
      "x2": 3684,
      "y2": 1706,
      "nuclei_count": 15
    },
    {
      "x1": 1228,
      "y1": 1706,
      "x2": 2456,
      "y2": 2560,
      "nuclei_count": 14
    },
    {
      "x1": 1228,
      "y1": 853,
      "x2": 2456,
      "y2": 1706,
      "nuclei_count": 14
    },
    {
      "x1": 0,
      "y1": 0,
      "x2": 1228,
      "y2": 853,
      "nuclei_count": 9
    },
    {
  

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/drive/MyDrive/pathrag_assets/histocartography_checkpoints")
dst = Path("/root/.local/share/pathrag/histocartography/checkpoints")
dst.mkdir(parents=True, exist_ok=True)

for name in ["pannuke.pt", "monusac.pt"]:
    shutil.copy2(src / name, dst / name)

print(list(dst.glob("*")))


[PosixPath('/root/.local/share/pathrag/histocartography/checkpoints/pannuke.pt'), PosixPath('/root/.local/share/pathrag/histocartography/checkpoints/monusac.pt')]


In [ ]:
from pathlib import Path
import sys
import os

chief_repo = Path("/root/.local/share/pathrag/chief/repo")
print("exists:", chief_repo.exists())
print("chief_heatmap exists:", (chief_repo / "chief_heatmap.py").exists())

sys.path.insert(0, str(chief_repo))
os.environ["MPLBACKEND"] = "Agg"

from chief_heatmap import extract_top_k_patches
print("import ok")


exists: True
chief_heatmap exists: True


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
%%bash
set -euo pipefail
rm -rf /content/envs/chief39


In [ ]:
%%bash
set -euo pipefail

/content/bin/micromamba run -p /content/envs/chief39 pip install --no-cache-dir \
  --index-url https://download.pytorch.org/whl/cu124 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

/content/bin/micromamba run -p /content/envs/chief39 pip install --no-cache-dir \
  numpy==1.26.4 \
  pillow==11.3.0 \
  matplotlib==3.9.4 \
  openslide-python==1.4.2 \
  timm==0.5.4 \
  addict==2.4.0 \
  PyYAML==6.0.3 \
  tqdm==4.67.3 \
  rich==15.0.0 \
  typer==0.23.2 \
  httpx==0.28.1 \
  huggingface_hub==1.8.0


Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.3/908.3 MB 313.9 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 277.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 316.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 288.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 596.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 331.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 324.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 264.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 290.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 293.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 325.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 241

In [ ]:
import subprocess, textwrap

cmd = [
    "/content/envs/chief39/bin/python",
    "-c",
    textwrap.dedent("""
        import numpy
        import matplotlib
        import PIL
        import openslide
        import timm
        import torch
        print("numpy", numpy.__version__)
        print("matplotlib", matplotlib.__version__)
        print("pillow ok")
        print("openslide ok")
        print("timm", timm.__version__)
        print("torch", torch.__version__)
    """),
]
r = subprocess.run(cmd, capture_output=True, text=True)
print("returncode:", r.returncode)
print("STDOUT:\\n", r.stdout)
print("STDERR:\\n", r.stderr)


returncode: 0
STDOUT:\n numpy 1.26.4
matplotlib 3.9.4
pillow ok
openslide ok
timm 0.5.4
torch 2.5.1+cu124

STDERR:\n 


In [ ]:
%%bash
set -euo pipefail
/content/bin/micromamba create -y -p /content/envs/chief39 python=3.9 pip
/content/bin/micromamba run -p /content/envs/chief39 python --version


conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache


Transaction

  Prefix: /content/envs/chief39

  Updating specs:

   - python=3.9
   - pip


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + bzip2                 1.0.8  hda65f42_9            conda-forge     Cached
  + ca-certificates   2026.4.22  hbd8a1cb_0            conda-forge     Cached
  + icu                    78.3  h33c6efd_0            conda-forge     Cached
  + ld_impl_linux-64     2.45.1  default_hbd61a6d_102  conda-forge     Cached
  + libexpat              2.7.5  hecca717_0            conda-forge     Cached
  + libffi                3.5.2  h3435931_0 

In [ ]:
import subprocess, textwrap

cmd = [
    "/content/envs/chief39/bin/python",
    "-c",
    textwrap.dedent("""
        import numpy
        import matplotlib
        import PIL
        import openslide
        import timm
        import torch
        print("numpy", numpy.__version__)
        print("matplotlib", matplotlib.__version__)
        print("pillow ok")
        print("openslide ok")
        print("timm", timm.__version__)
        print("torch", torch.__version__)
    """),
]
r = subprocess.run(cmd, capture_output=True, text=True)
print("returncode:", r.returncode)
print("STDOUT:\\n", r.stdout)
print("STDERR:\\n", r.stderr)


returncode: 0
STDOUT:\n numpy 1.26.4
matplotlib 3.9.4
pillow ok
openslide ok
timm 0.5.4
torch 2.5.1+cu124

STDERR:\n 


In [ ]:
import subprocess, textwrap

code = textwrap.dedent("""
from pathlib import Path
import sys, os

chief_repo = Path("/root/.local/share/pathrag/chief/repo")
sys.path.insert(0, str(chief_repo))
os.environ["MPLBACKEND"] = "Agg"

from chief_heatmap import extract_top_k_patches
print("import ok")
""")

cmd = ["/content/envs/chief39/bin/python", "-c", code]
r = subprocess.run(cmd, capture_output=True, text=True)
print("returncode:", r.returncode)
print("STDOUT:\\n", r.stdout)
print("STDERR:\\n", r.stderr)


returncode: 0
STDOUT:\n import ok

STDERR:\n 


In [ ]:
import json
import os
import subprocess
from pathlib import Path

hc = json.loads(Path("/content/hc_smoke.json").read_text())
valid_bounds = [
    [p["x1"], p["y1"], p["x2"], p["y2"]]
    for p in hc.get("selected_patches", [])[:TOP_K]
]

chief_cmd = [
    os.environ["CHIEF_PYTHON"],
    "-m", "src.graph.chief_eval",
    "--config", str(STARTER_DIR / "tools" / "chief" / "config" / "default.yaml"),
    "--image-path", IMAGE_PATH,
    "--top-k", str(TOP_K),
    "--valid-bounds-json", json.dumps(valid_bounds),
    "--out", "/content/chief_smoke.json",
]
subprocess.run(chief_cmd, cwd=str(STARTER_DIR / "tools" / "chief"), check=True)
print(Path("/content/chief_smoke.json").read_text()[:2000])


{
  "top_k_patches": [
    {
      "x1": 1344,
      "y1": 448,
      "x2": 1568,
      "y2": 672
    },
    {
      "x1": 672,
      "y1": 2016,
      "x2": 896,
      "y2": 2240
    },
    {
      "x1": 448,
      "y1": 1792,
      "x2": 672,
      "y2": 2016
    }
  ],
  "heatmap": [
    [
      0.5210134983062744,
      0.537438154220581,
      0.3619086444377899,
      0.587432324886322,
      0.5977531671524048,
      0.33081740140914917,
      0.3521605432033539,
      0.5558578968048096,
      0.479364275932312,
      0.47714000940322876,
      0.4851529002189636,
      0.5028610229492188,
      0.6962324976921082,
      0.5898205637931824,
      0.6553124785423279,
      0.4357456564903259,
      0.03829453885555267
    ],
    [
      0.47785282135009766,
      0.6030019521713257,
      0.4215604066848755,
      0.9040260314941406,
      0.8341217041015625,
      0.7670133709907532,
      0.16224530339241028,
      0.5724472999572754,
      0.3390510380268097,
      0.31866568

In [ ]:
%%bash
set -euo pipefail
/content/bin/micromamba run -p /content/envs/llava310 pip install shortuuid
%%bash


In [ ]:
%%bash
set -euo pipefail
/content/bin/micromamba run -p /content/envs/llava310 pip install \
  shortuuid \
  sentencepiece \
  protobuf \
  einops \
  einops-exts \
  timm \
  accelerate


In [ ]:
import json
from pathlib import Path
import sys

sys.path.insert(0, str(STARTER_DIR / "src"))
from pathrag.agents.langgraph_app import build_graph

app = build_graph()
init = {
    "image_path": IMAGE_PATH,
    "question": QUESTION,
    "mode": "answer",
    "top_k": TOP_K,
    "max_rounds": MAX_ROUNDS,
    "round_ix": 0,
}

merged = {}
for state in app.stream(init, config={"configurable": {"thread_id": "colab-full-run"}}):
    if isinstance(state, dict):
        merged.update(state)

out_path = Path("/content/pathrag_full_run.json")
out_path.write_text(json.dumps(merged, indent=2), encoding="utf-8")
print(f"Saved full run to {out_path}")

[INFO] 2026-04-24 05:45:48,121 pathrag.langgraph: Stage 1–2 start
[INFO] 2026-04-24 05:45:48,125 pathrag.tools: Tiled image /content/drive/MyDrive/pathrag_inputs/07_level2.jpg into 9 patches.
[INFO] 2026-04-24 05:46:02,019 pathrag.tools: Ranked 9 patches by local histocartography (selected=3, non_selected=6).
[INFO] 2026-04-24 05:46:10,150 pathrag.tools: Stage 1–2 sequential handoff: 3 HC patches -> 3 valid bounds -> 3 CHIEF patches
[INFO] 2026-04-24 05:46:10,151 pathrag.langgraph: Stage 1–2 done: hc=9 chief=3 final=3
[INFO] 2026-04-24 05:46:10,153 pathrag.langgraph: Stage 3 start
[INFO] 2026-04-24 05:46:10,153 pathrag.langgraph: Stage 3 done: label=head_neck, caps=2
[INFO] 2026-04-24 05:46:10,155 pathrag.langgraph: Stage 4 start
[INFO] 2026-04-24 05:49:04,892 pathrag.langgraph: Stage 4 done: patches=3
[INFO] 2026-04-24 05:49:04,895 pathrag.langgraph: Stage 5 round 0 start
[INFO] 2026-04-24 05:49:07,267 pathrag.langgraph: Stage 5 round done → round_ix=1
[INFO] 2026-04-24 05:49:07,270 p

In [ ]:
import json
from pathlib import Path

result = json.loads(Path("/content/pathrag_full_run.json").read_text())

stage12 = result.get("tile_rank", result)
stage3 = result.get("identify", result)
stage4 = result.get("stage4", result)
stage6 = result.get("rerank", result)
stage7 = result.get("fuse", result)

print("Final answer:\n")
print(stage7.get("final_answer", result.get("final_answer", "")))
print("\nHC selected patches:")
print(json.dumps(stage12.get("hc_rank", result.get("hc_rank", [])), indent=2)[:2000])
print("\nCHIEF patches:")
print(json.dumps(stage12.get("patches", result.get("patches", [])), indent=2)[:2000])
print("\nChosen idx:", stage6.get("chosen_idx", result.get("chosen_idx", [])))
print("\nROI descriptions:")
print(json.dumps(stage4.get("roi_desc", result.get("roi_desc", [])), indent=2)[:2000])
print("\nPatch summaries:")
print(json.dumps(stage4.get("patch_summaries", result.get("patch_summaries", [])), indent=2)[:3000])


Final answer:

Based on the evidence from the selected patches (CP0, CP1, CP2), the prominent, discrete, thick-walled circular structures with clear or collapsed lumens embedded within the looser pale pink stroma are more consistent with blood vessels. Each patch consistently identifies the dominant structures as blood vessels, which are characterized by their thick walls and lumens that may appear clear or collapsed depending on the sectioning and preparation of the tissue.

There is no evidence from the patches to suggest any other type of structure, such as glandular or cystic formations, that might be considered in the context of head and neck pathology. Therefore, based solely on the patch evidence, these structures are best identified as blood vessels. If further differentiation or diagnosis is required, additional evidence or context would be necessary.

HC selected patches:
[
  {
    "id": "P0",
    "bbox": [
      1228,
      0,
      2456,
      853
    ],
    "score": 18.0
 

## Notes

- If the full run fails in Stage 4, debug the `llava-med` environment first. The most common issues are Hugging Face access, model OOM, or an incompatible repo checkout.
- If Stage 1 fails, check that:
  - `~/.local/share/pathrag/chief/repo` contains the CHIEF source tree
  - `~/.local/share/pathrag/chief/model_weight` contains `CHIEF_CTransPath.pth`, `CHIEF_pretraining.pth`, and `Text_emdding.pth`
- Histocartography checkpoints download automatically on first use into `~/.local/share/pathrag/histocartography/checkpoints`.